# Sigmoid & Sigmoid Derivative

The sigmoid function is one of the most important building blocks in neural networks.
It squashes any real number into a value between 0 and 1.

## 1. Sigmoid Function

**Formula:**

```
σ(x) = 1 / (1 + e^(-x))
```

**What it does:** Maps any input to a value between 0 and 1.

**When to use:**
- Binary classification (yes/no, spam/not spam) — output acts like a probability
- Neural network activation function — decides how much a neuron "fires"
- Logistic regression

In [ ]:
import math

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

# Test with different inputs
test_values = [-10, -5, -1, 0, 1, 5, 10]

print("Sigmoid Function")
print("-" * 35)
print(f"{'Input':>8} | {'Output':>10}")
print("-" * 35)
for x in test_values:
    print(f"{x:>8} | {sigmoid(x):>10.6f}")

### Key observations

- `sigmoid(0) = 0.5` — the midpoint
- Negative inputs → closer to 0
- Positive inputs → closer to 1
- The output **never** actually reaches 0 or 1, only approaches them

### ASCII visualization

In [ ]:
# ASCII plot of sigmoid curve
width = 60
print("\nSigmoid Curve (x from -6 to 6)")
print("=" * (width + 15))

for row in range(20, -1, -1):
    y_level = row / 20  # 0.0 to 1.0
    line = ""
    for col in range(width):
        x = -6 + 12 * col / (width - 1)  # -6 to 6
        y = sigmoid(x)
        if abs(y - y_level) < 0.03:
            line += "*"
        elif col == width // 2:
            line += "|"
        elif row == 10:
            line += "-"
        else:
            line += " "
    label = f" {y_level:.1f}" if row % 5 == 0 else ""
    print(f"{line}{label}")

print(" " * (width // 2 - 3) + "-6    0    6")

## 2. Sigmoid Derivative

**Formula:**

```
σ'(x) = σ(x) × (1 - σ(x))
```

**What it does:** Tells you the **slope** (rate of change) of the sigmoid at any point.

**When to use:**
- **Backpropagation** — calculating how much to adjust each weight
- **Gradient descent** — the derivative tells the network which direction to move and how far

The elegant part: the derivative is expressed entirely in terms of sigmoid itself.

In [ ]:
def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

print("Sigmoid Derivative")
print("-" * 45)
print(f"{'Input':>8} | {'sigmoid(x)':>10} | {'derivative':>10}")
print("-" * 45)
for x in test_values:
    s = sigmoid(x)
    d = sigmoid_derivative(x)
    print(f"{x:>8} | {s:>10.6f} | {d:>10.6f}")

### Key observations

- Maximum derivative at `x = 0` → `0.25` (steepest slope)
- At the extremes (`x = ±10`), derivative is near `0` (curve is flat)
- This means: when the neuron is very confident (close to 0 or 1), learning slows down

### ASCII visualization: sigmoid vs derivative side by side

In [ ]:
width = 50
print("\nSigmoid (S) vs Derivative (D)")
print("x from -6 to 6")
print("=" * (width + 10))

for row in range(20, -1, -1):
    y_level = row / 20  # 0.0 to 1.0
    line = ""
    for col in range(width):
        x = -6 + 12 * col / (width - 1)
        y_sig = sigmoid(x)
        y_der = sigmoid_derivative(x)
        if abs(y_sig - y_level) < 0.03 and abs(y_der - y_level) < 0.03:
            line += "X"  # overlap
        elif abs(y_sig - y_level) < 0.03:
            line += "S"
        elif abs(y_der - y_level) < 0.03:
            line += "D"
        elif col == width // 2:
            line += "|"
        else:
            line += " "
    label = f" {y_level:.1f}" if row % 5 == 0 else ""
    print(f"{line}{label}")

print(f"\n  S = sigmoid    D = derivative    X = overlap")

## 3. How They Work Together in a Neural Network

During training:

1. **Forward pass** — `sigmoid(x)` produces the prediction
2. **Backward pass** — `sigmoid_derivative(x)` tells us how to adjust weights

Think of it like driving:
- **Sigmoid** = your position on the road (between 0 and 1)
- **Derivative** = your speed (how fast you're moving)

In [ ]:
# Simple neuron example: forward and backward pass

# A single neuron with 3 inputs
inputs = [0.5, 0.8, -0.2]
weights = [0.4, -0.3, 0.7]
bias = 0.1

# --- Forward Pass ---
weighted_sum = sum(i * w for i, w in zip(inputs, weights)) + bias
output = sigmoid(weighted_sum)

print("=== Forward Pass ===")
print(f"Inputs:       {inputs}")
print(f"Weights:      {weights}")
print(f"Bias:         {bias}")
print(f"Weighted sum: {weighted_sum:.4f}")
print(f"Output:       sigmoid({weighted_sum:.4f}) = {output:.4f}")

# --- Backward Pass ---
target = 1.0  # what we wanted
error = target - output
gradient = error * sigmoid_derivative(weighted_sum)

print(f"\n=== Backward Pass ===")
print(f"Target:       {target}")
print(f"Error:        {target} - {output:.4f} = {error:.4f}")
print(f"Derivative:   sigmoid'({weighted_sum:.4f}) = {sigmoid_derivative(weighted_sum):.4f}")
print(f"Gradient:     {error:.4f} × {sigmoid_derivative(weighted_sum):.4f} = {gradient:.4f}")

# --- Weight Update ---
learning_rate = 0.5
new_weights = [w + learning_rate * gradient * i for w, i in zip(weights, inputs)]

print(f"\n=== Weight Update (lr={learning_rate}) ===")
for i, (old, new) in enumerate(zip(weights, new_weights)):
    print(f"  w{i}: {old:.4f} → {new:.4f}  (Δ = {new - old:+.4f})")

## 4. The Vanishing Gradient Problem

The sigmoid derivative has a maximum value of `0.25`.
In deep networks, gradients are **multiplied** through each layer.

If each layer multiplies by at most 0.25:

In [ ]:
# Vanishing gradient: multiply max derivative across layers
max_derivative = 0.25

print("Vanishing Gradient Problem")
print("-" * 45)
print(f"{'Layers':>8} | {'Gradient multiplier':>20} | {'Status'}")
print("-" * 45)

gradient = 1.0
for layer in range(1, 11):
    gradient *= max_derivative
    if gradient > 0.01:
        status = "learning"
    elif gradient > 0.0001:
        status = "struggling"
    else:
        status = "effectively dead"
    print(f"{layer:>8} | {gradient:>20.10f} | {status}")

print("\nThis is why deep networks prefer ReLU over sigmoid.")
print("ReLU derivative = 1 (for positive inputs), so gradients don't shrink.")

## 5. NumPy Version (Vectorized)

In practice, we use NumPy to apply sigmoid to entire arrays at once.

In [ ]:
import numpy as np

def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative_np(x):
    s = sigmoid_np(x)
    return s * (1 - s)

# Apply to an entire array at once
x = np.array([-5, -2, -1, 0, 1, 2, 5])

print("NumPy vectorized sigmoid")
print(f"Input:      {x}")
print(f"Sigmoid:    {np.round(sigmoid_np(x), 4)}")
print(f"Derivative: {np.round(sigmoid_derivative_np(x), 4)}")

## 6. Sigmoid in Modern LLMs — Not Gone, Just Specialized

ReLU (and its descendants) replaced sigmoid as the **general-purpose** activation in hidden layers.
But sigmoid is still very much alive — it moved to more specialized, important roles.

### Where sigmoid still lives

| Role | Example | Why sigmoid? |
|---|---|---|
| **Gating** | SwiGLU (LLaMA, Mistral), LSTM gates | Need a 0-1 value to control how much signal passes |
| **Binary output** | Spam detection, sentiment | Need a probability |
| **Routing** | Mixture-of-Experts | Need a soft yes/no per expert |

### The evolution of hidden-layer activations

```
Sigmoid → ReLU → GELU → SwiGLU
(2000s)   (2012)  (2016)  (2020+)
```

Interestingly, **SwiGLU** — the activation used in today's best LLMs — has sigmoid *inside* it:

```
SwiGLU(x) = x × σ(x) × W
              ↑
          sigmoid is here, acting as a gate
```

## Summary

| | Sigmoid | Sigmoid Derivative |
|---|---|---|
| **Formula** | `1 / (1 + e^(-x))` | `σ(x) × (1 - σ(x))` |
| **Output range** | (0, 1) | (0, 0.25] |
| **Used in** | Forward pass | Backward pass |
| **Purpose** | Squash to probability | Calculate weight updates |
| **Max at** | approaches 1 | x = 0 (value = 0.25) |
| **Weakness** | Saturates at extremes | Vanishing gradients |

In [ ]:
# Sigmoid's modern roles: gating functions

def swish(x):
    """x * sigmoid(x) — the core of SwiGLU"""
    return x * sigmoid(x)

def gelu_approx(x):
    """Gaussian Error Linear Unit (used in BERT, GPT-2)"""
    return 0.5 * x * (1 + math.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)))

print("Modern Activations (all evolved from sigmoid + ReLU ideas)")
print("-" * 65)
print(f"{'x':>6} | {'Sigmoid':>8} | {'ReLU':>6} | {'Swish':>8} | {'GELU':>8}")
print("-" * 65)

for x in [-3, -2, -1, -0.5, 0, 0.5, 1, 2, 3]:
    print(f"{x:>6.1f} | {sigmoid(x):>8.4f} | {max(0,x):>6.1f} | {swish(x):>8.4f} | {gelu_approx(x):>8.4f}")

print("\nNotice how Swish and GELU behave like:")
print("  - ReLU for large positive values (pass through)")
print("  - Near 0 for large negative values (block)")
print("  - Smooth transition around 0 (unlike ReLU's sharp corner)")
print("  - Slightly negative dip (unlike ReLU's hard zero)")

In [ ]:
# How sigmoid acts as a gate inside Swish

print("Inside Swish: x × sigmoid(x)")
print("="* 55)
print(f"{'x':>6} | {'sigmoid(x)':>10} | {'x × σ(x)':>10} | Gate effect")
print("-" * 55)

for x in [-5, -3, -1, 0, 1, 3, 5]:
    s = sigmoid(x)
    result = x * s
    if s < 0.1:
        effect = f"gate {s:.0%} closed → blocks"
    elif s > 0.9:
        effect = f"gate {s:.0%} open → passes"
    else:
        effect = f"gate {s:.0%} open → partial"
    print(f"{x:>6} | {s:>10.4f} | {result:>10.4f} | {effect}")

print("\nSigmoid's job: decide how much of x to let through.")
print("This is the same gating idea used in LSTM and Transformers.")

### Bottom line

- **Sigmoid** didn't disappear — it got promoted from grunt work to a specialist role
- As a **general activation** (hidden layers): replaced by ReLU → GELU → SwiGLU
- As a **gate** (controlling signal flow): still sigmoid, still irreplaceable
- As a **probability output** (binary classification): still sigmoid